<img src="logos/Logo_de_la_Facultad_Experimental_de_Ciencia_y_Tecnología.svg.png" width="250" style="display: block; margin-left: auto; margin-right: auto;">

# Premier League — Model Selection y Optimización de Hiperparámetros

## Propósito

Notebook final de modelado: comparar modelos de forma justa, fijar el protocolo de evaluación, decidir el tuning de hiperparámetros y evaluar una sola vez el pipeline final sobre test.

> **Entregable único:** este notebook ejecutado de principio a fin (Run All desde kernel limpio). No se entrega informe, presentación ni modelo serializado.

> **Estado:** secciones 1–3 implementadas (sin entrenamiento). Las secciones 4–12 quedan como esqueleto.

## 1. Objetivo, métrica y protocolo de evaluación

El problema es una **clasificación de 3 clases** con dominio cerrado `{H, D, A}` (una fila por partido), sobre `FTR`. La partición temporal ya está fijada por el notebook 02: cuatro temporadas para train y la temporada 2025-26 reservada para una única evaluación final.

**Métrica principal: log-loss (menor es mejor).** Castiga por dar poca probabilidad a lo que sí pasó y premia al repartir la probabilidad con el acercamiento mas acertado. Es el error que importa aquí: un modelo que dice 80 % local cuando el partido termina en empate no es lo mismo que uno que repartió 45 % local / 45 % empate, aunque ambos fallen el `argmax`. También es la métrica con la que se puede comparar directamente contra las probabilidades del mercado.

**Métricas secundarias**, cada una con una pregunta distinta:

- **Accuracy:** ¿supera la referencia trivial (siempre local) y la del mercado (favorito de-vig)?
- **Balanced accuracy:** ¿rinde parejo en las tres clases, incluido el empate?
- **Matriz de confusión:** ¿qué confunde con qué?
- **Calibración:** cuando declara 60 %, ¿ocurre cerca del 60 %?

**Regla de decisión, escrita antes de ver resultados:**

1. gana el **menor log-loss medio** en validación cruzada;
2. si dos pipelines quedan dentro de **una desviación estándar**, se consideran empatados;
3. en empate: menor desviación entre folds y luego menor costo/complejidad;
4. la clase predicha sale del `argmax` de las probabilidades, sin umbrales ajustados a mano;
5. test se usa **una sola vez**, después de cerrar la selección.

In [1]:
# Contrato de evaluación fijado antes de ver resultados.
CLASSES = ["H", "D", "A"]
PRIMARY_METRIC = "log_loss"  # menor es mejor
SECONDARY_METRICS = (
    "accuracy",           # ¿supera la referencia trivial y la del mercado?
    "balanced_accuracy",  # ¿rinde parejo en H/D/A?
    "confusion_matrix",   # ¿qué confunde con qué?
    "calibration",        # ¿un 60 % declarado ocurre cerca del 60 %?
)
TIE_WITHIN_STD = 1.0  # dentro de una desviación estándar se considera empate
TEST_USED_ONCE = True

## 2. Recuperar el pipeline del notebook 02

El notebook 02 no serializa artefactos: su conclusión indica que el modelado reconstruye los objetos reejecutando sus mismas celdas. Por eso esta sección copia **sin cambios** las piezas que definen el contrato, en este orden:

1. imports, rutas y constantes;
2. validación del esquema y de los rangos;
3. corrección de la anomalía y partición temporal por fecha;
4. `market_probabilities` y las dos referencias;
5. `build_features` y sus funciones auxiliares;
6. `build_preprocessor` y `build_model_pipeline`.

Único cambio deliberado: `RANDOM_STATE = 777` (decisión del equipo). En el 02 ese valor no tenía efecto porque no se muestreaba; aquí sólo lo consumirán los estimadores estocásticos, porque los folds temporales no usan azar.

Nada que aprenda estadísticas se ajusta fuera de un fold: imputación, escalado y one-hot viven dentro de `build_model_pipeline`. `build_features` es pura y causal, y se calcula una vez sobre el marco cronológico completo para luego cortar por índice.

Las comprobaciones al final de la sección son el contrato ejecutable: si el 02 cambia, esto falla en vez de continuar en silencio.

### 2.1 Entorno, rutas y semilla

Se fijan versiones, rutas y la semilla global. La semilla solo la consumen los estimadores con azar; los folds temporales son deterministas, así que no dependen de ella. La salida confirma el intérprete y el CSV que se va a usar.

In [2]:
from __future__ import annotations

import platform
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sklearn
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# Semilla global del modelado (decisión del equipo: 777). Los folds
# temporales no usan azar; solo la consumen los estimadores estocásticos.
RANDOM_STATE = 777
TARGET_COLUMN = "FTR"
CUTOFF = pd.Timestamp("2025-08-01")
# Sliding Window Size
K = 5

PROJECT_ROOT = Path.cwd()

if not (PROJECT_ROOT / "data" / "processed").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_PATH = (
    PROJECT_ROOT / "data" / "processed"
    / "premier_league_2021_2026.csv"
)

print(f"Python: {platform.python_version()}")
print(f"pandas: {pd.__version__}")
print(f"scikit-learn: {sklearn.__version__}")


Python: 3.14.6
pandas: 3.0.5
scikit-learn: 1.9.1


### 2.2 Contrato de columnas

Se declara la *whitelist* positiva: solo estas columnas pueden entrar en `X`. Las columnas post-partido quedan fuera, salvo `FTHG`/`FTAG`, que solo se usan como resultados de partidos anteriores. Aquí también se fijan las 23 features y los tamaños esperados de train y test.

In [3]:
REQUIRED_COLUMNS = [
    "Season",
    "Date",
    "Time",
    "HomeTeam",
    "AwayTeam",
    "FTR",
    "FTHG",
    "FTAG",
    "AvgH",
    "AvgD",
    "AvgA",
    "AHh",
]

# The 17 columns that only exist after the final whistle. FTHG and FTAG are
# the deliberate exception: they enter as the results of *earlier* matches.
POST_MATCH_COLUMNS = [
    "FTHG",
    "FTAG",
    "HTHG",
    "HTAG",
    "HTR",
    "HS",
    "AS",
    "HST",
    "AST",
    "HF",
    "AF",
    "HC",
    "AC",
    "HY",
    "AY",
    "HR",
    "AR",
]

# The 42 exchange 1X2 quote columns. This is the only block of the dataset
# where absence is structural, so it is the only place where an availability
# indicator has non-zero variance.
ODDS_QUOTE_COLS = [
    "BFH", "BFD", "BFA", "1XBH", "1XBD", "1XBA",
    "BFEH", "BFED", "BFEA", "BFDH", "BFDD", "BFDA",
    "BFCH", "BFCD", "BFCA", "1XBCH", "1XBCD", "1XBCA",
    "BFDCH", "BFDCD", "BFDCA",
    "BFECH", "BFECD", "BFECA",
    "BMGMH", "BMGMD", "BMGMA", "BMGMCH", "BMGMCD", "BMGMCA",
    "CLH", "CLD", "CLA", "CLCH", "CLCD", "CLCA",
    "LBH", "LBD", "LBA", "LBCH", "LBCD", "LBCA",
]

# Positive whitelist. Subtracting a list is as fragile as adding one.
ALLOWED_RAW_COLUMNS = (
    ["Season", "Date", "Time", "HomeTeam", "AwayTeam",
     "FTHG", "FTAG",
     "AvgH", "AvgD", "AvgA", "AHh"]
    + ODDS_QUOTE_COLS
)

# The features that survive the redundancy pruning of the catalogue.
HIST_KEEP = [
    "Home_Hist_Pts_k",
    "Away_Hist_Pts_k",
    "Diff_Hist_Pts_k",
    "Home_Hist_GF_k",
    "Away_Hist_GF_k",
    "Diff_Hist_GF_k",
    "Home_Hist_GA_k",
    "Away_Hist_GA_k",
    "Home_Hist_W_k",
    "Away_Hist_W_k",
    "Home_Hist_L_k",
    "Away_Hist_L_k",
]
MARKET_KEEP = ["p_home", "p_draw", "p_away", "Mkt_Gap", "AHh"]
CAL_KEEP = ["PriorHomeMatches", "PriorAwayMatches", "OddsQuotes"]
DOW_KEEP = ["DayOfWeek"]
BASE_FEATURES = HIST_KEEP + MARKET_KEEP + CAL_KEEP + DOW_KEEP

# Raw identity columns kept so the one-hot block can consume them.
CLUB_COLS = ["HomeTeam", "AwayTeam"]
PIPELINE_FEATURES = BASE_FEATURES + CLUB_COLS


# Verified expectations used by the validation and split assertions.
MATCHES_PER_SEASON = 380
TRAIN_ROWS = 1520
TEST_ROWS = 380
# Corrected total shots for Newcastle - West Ham, 15/08/2021, where
# shots on target exceeded total shots; verified against an external
# source.
CORRECTED_AS = 18


### 2.3 Carga y validación del esquema

Antes de construir nada se comprueba que el CSV tiene las columnas requeridas, que `FTR` solo contiene `H/D/A`, que no hay equipos repetidos y que cada temporada tiene 380 partidos. Si el archivo cambia, la celda se detiene con un mensaje en vez de seguir con datos rotos.

In [4]:
df = pd.read_csv(DATA_PATH)

missing_columns = [
    column for column in REQUIRED_COLUMNS if column not in df.columns
]

if missing_columns:
    raise ValueError(f"Missing required columns: {missing_columns}")

if not df[TARGET_COLUMN].isin(["H", "D", "A"]).all():
    raise ValueError("FTR must contain only H, D or A.")

if (df["HomeTeam"] == df["AwayTeam"]).any():
    raise ValueError("HomeTeam must differ from AwayTeam.")

primary_key = ["Season", "HomeTeam", "AwayTeam"]

if df.duplicated(subset=primary_key).any():
    raise ValueError(f"{primary_key} must be unique.")

if not df.groupby("Season").size().eq(MATCHES_PER_SEASON).all():
    raise ValueError(
        f"Every season must contain {MATCHES_PER_SEASON} matches."
    )

print(f"Validated dataset: {df.shape[0]} rows and {df.shape[1]} columns")
df.head()

Validated dataset: 1900 rows and 157 columns


,Season,Div,Date,Time,HomeTeam,AwayTeam,FTHG,FTAG,FTR,HTHG,...,BFDCA,BMGMCH,BMGMCD,BMGMCA,CLCH,CLCD,CLCA,LBCH,LBCD,LBCA
0,2021-2022,E0,13/08/2021,20:00,Brentford,Arsenal,2,0,H,1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2021-2022,E0,14/08/2021,12:30,Man United,Leeds,5,1,H,1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2021-2022,E0,14/08/2021,15:00,Burnley,Brighton,1,2,A,1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2021-2022,E0,14/08/2021,15:00,Chelsea,Crystal Palace,3,0,H,2,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2021-2022,E0,14/08/2021,15:00,Everton,Southampton,3,1,H,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


La salida debe ser **1900 filas × 157 columnas**: cinco temporadas completas. Cualquier otro tamaño indica que el CSV de entrada no es el esperado.

### 2.4 Rangos y categorías

Se verifican valores físicamente posibles: cuotas por encima de 1.00, goles no negativos, sin infinitos y con `Div = E0`. Son violaciones de dominio, no outliers: un valor imposible invalida la fila.

In [5]:
if set(df["Div"].unique()) != {"E0"}:
    raise ValueError(f"Div must be E0, found {set(df['Div'].unique())}")

if not df["Season"].str.match(r"^\d{4}-\d{4}$").all():
    raise ValueError("Season must look like 2021-2022.")

if not df["Time"].str.match(r"^\d{2}:\d{2}$").all():
    raise ValueError("Time must look like 15:00.")

odds_columns = ["AvgH", "AvgD", "AvgA", "MaxH", "MaxD", "MaxA"]

if df[odds_columns].le(1.0).any().any():
    raise ValueError(f"Bookmaker odds must be above 1.0: {odds_columns}")

goal_columns = ["FTHG", "FTAG", "HTHG", "HTAG"]

if df[goal_columns].lt(0).any().any():
    raise ValueError(f"Goals cannot be negative: {goal_columns}")

numeric_block = df.select_dtypes(include="number").to_numpy(dtype=float)

if np.isinf(numeric_block).any():
    raise ValueError("Infinite values in the numeric block.")

nan_cells = int(np.isnan(numeric_block).sum())

print("Impossible ranges and invalid categories: none found.")
print(f"All numeric values finite; {nan_cells} missing cells recorded.")

Impossible ranges and invalid categories: none found.
All numeric values finite; 86394 missing cells recorded.


### 2.5 Corrección reproducible

El EDA detectó una sola fila incoherente (`AST > AS`). Se localiza por fecha y equipos, se exige que coincida exactamente una fila y se corrige `AS`. Después se comprueban los invariantes de tiros para que la corrección no oculte otro problema.

In [6]:
df["Date"] = pd.to_datetime(df["Date"], format="%d/%m/%Y")

match_key = (
    (df["Date"] == "2021-08-15")
    & (df["HomeTeam"] == "Newcastle")
    & (df["AwayTeam"] == "West Ham")
)

if int(match_key.sum()) != 1:
    raise ValueError(
        "Expected exactly 1 Newcastle-West Ham row, got "
        f"{int(match_key.sum())}"
    )

df.loc[match_key, "AS"] = CORRECTED_AS

if not df["AST"].le(df["AS"]).all():
    raise ValueError("AST must not exceed AS.")

if not df["HST"].le(df["HS"]).all():
    raise ValueError("HST must not exceed HS.")

print("Anomaly corrected and the shot invariants hold.")

Anomaly corrected and the shot invariants hold.


### 2.6 Corte temporal

Train y test se separan por fecha (`2025-08-01`), no al azar: se quiere predecir una temporada no vista. La celda comprueba que no hay filas compartidas, que train termina antes del corte y que los tamaños son 1520 y 380.

In [7]:
is_test = df["Date"] >= CUTOFF
df_train = df.loc[~is_test].copy()
df_test = df.loc[is_test].copy()

assert len(df_train) + len(df_test) == len(df)
assert set(df_train.index).isdisjoint(df_test.index)
assert df_train["Date"].max() < df_test["Date"].min()
assert df_train["Season"].nunique() == 4
assert df_test["Season"].nunique() == 1
assert (len(df_train), len(df_test)) == (TRAIN_ROWS, TEST_ROWS)
assert df_train["Date"].max() < CUTOFF <= df_test["Date"].min()

def period(frame):
    """Return the first and last match date as a readable string."""
    start = frame["Date"].min()
    end = frame["Date"].max()
    return f"{start:%Y-%m-%d} to {end:%Y-%m-%d}"


print(f"Train rows: {len(df_train)} ({period(df_train)})")
print(f"Test rows: {len(df_test)} ({period(df_test)})")

Train rows: 1520 (2021-08-13 to 2025-05-25)
Test rows: 380 (2025-08-15 to 2026-05-24)


La partición respeta el orden real del calendario. El test queda cerrado desde aquí: solo se usará en la evaluación final.

### 2.7 De-vig y referencias

`market_probabilities` convierte las cuotas 1X2 en probabilidades que suman 1 (de-vig proporcional). Con ellas se calculan las dos referencias del proyecto: la clase mayoritaria y el favorito del mercado. No son modelos entrenados; son los pisos que el pipeline debe superar.

In [8]:
def market_probabilities(frame, cols=("AvgH", "AvgD", "AvgA")):
    """Return de-vigged 1X2 probabilities, one row per match.

    Applies the proportional de-vig p_i = (1 / o_i) / sum_j (1 / o_j),
    assuming the bookmaker margin splits equally across the outcomes.

    Parameters
    ----------
    frame : pandas.DataFrame
        Match frame with the three 1X2 quote columns.
    cols : tuple of str
        Quote columns, in home, draw, away order.

    Returns
    -------
    numpy.ndarray
        One probability per outcome, shape (len(frame), 3); each row
        sums to 1.
    """
    inverse = 1.0 / frame[list(cols)].to_numpy(dtype=float)
    return inverse / inverse.sum(axis=1, keepdims=True)


classes = np.array(["H", "D", "A"])
target_share = df[TARGET_COLUMN].value_counts(normalize=True)

print("Target distribution (n=1900)")
print((100 * target_share).round(2).rename("share %"))

trivial_accuracy = 100 * target_share.max()
print(
    f"\nTrivial baseline (always {target_share.idxmax()}): "
    f"{trivial_accuracy:.2f} %"
)

favorite = classes[market_probabilities(df).argmax(axis=1)]
market_accuracy = 100 * (favorite == df[TARGET_COLUMN].to_numpy()).mean()
print(f"Market baseline (de-vig + favourite): {market_accuracy:.2f} %")

Target distribution (n=1900)
FTR
H    44.16
A    31.95
D    23.89
Name: share %, dtype: float64

Trivial baseline (always H): 44.16 %
Market baseline (de-vig + favourite): 55.32 %


El favorito del mercado acierta más que la clase mayoritaria, pero nunca elige empate. Ese es el margen concreto donde un modelo puede aportar: cubrir la clase que el mercado ignora.

### 2.8 Bloque histórico

**Cómo se construye.** `team_match_long` convierte cada partido en dos filas: una desde la perspectiva del local y otra desde la del visitante. Dentro de cada `(Team, Season)` los partidos se ordenan por fecha, `shift(1)` retrasa los resultados una fila y `rolling(K, min_periods=1)` suma los últimos hasta 5 partidos. Después se vuelven a pegar las dos vistas y se calculan las diferencias. Así, la fila del partido que se predice nunca lee su propio resultado: solo ve lo anterior.

**Glosario:** `GF` = goles a favor, `GA` = goles en contra, `Pts` = puntos (3 victoria, 1 empate, 0 derrota), `W` = victorias, `L` = derrotas, `k` = ventana de `K = 5` partidos.

| Feature | Qué mide | Qué aporta |
| --- | --- | --- |
| `Home_Hist_Pts_k` | Puntos del equipo local en sus últimos ≤5 partidos | Forma reciente y capacidad de sumar |
| `Home_Hist_GF_k` | Goles a favor del local en esos partidos | Poder ofensivo |
| `Home_Hist_GA_k` | Goles en contra del local | Solidez defensiva |
| `Home_Hist_W_k` | Victorias del local | Frecuencia de cierre de partidos |
| `Home_Hist_L_k` | Derrotas del local | Tendencia a perder |
| `Away_Hist_Pts_k` | Puntos del equipo visitante | Forma reciente del rival |
| `Away_Hist_GF_k` | Goles a favor del visitante | Poder ofensivo del rival |
| `Away_Hist_GA_k` | Goles en contra del visitante | Solidez defensiva del rival |
| `Away_Hist_W_k` | Victorias del visitante | Frecuencia de victoria del rival |
| `Away_Hist_L_k` | Derrotas del visitante | Tendencia a perder del rival |
| `Diff_Hist_Pts_k` | `Home_Hist_Pts_k - Away_Hist_Pts_k` | Ventaja relativa de forma |
| `Diff_Hist_GF_k` | `Home_Hist_GF_k - Away_Hist_GF_k` | Ventaja relativa de ataque |

**Ojo con los prefijos.** `Home_` significa "del equipo que juega como local", no "solo en partidos de local"; su historial incluye también sus partidos de visitante. Lo mismo con `Away_`. `Diff_` es siempre local menos visitante: positivo favorece al local.

**Por qué no están todas.** `build_team_history` calcula también `Hist_D_k` y `Hist_GD_k`, pero `HIST_KEEP` no los conserva. `D` es redundante con `W` y `L` una vez conocido el número de partidos jugados, y `GD = GF - GA`; las 12 de la tabla son el corte final del notebook 02.

**Cómo leerlas.** Son **sumas** sobre la ventana, no promedios: al inicio de temporada hay menos de 5 partidos y el primer partido de cada equipo queda en `NaN` (se imputa con la mediana dentro del fold). La ventana se reinicia en cada temporada, así que no arrastra la forma del año anterior, y el `shift(1)` garantiza que no hay fuga de información.

In [9]:
# The 7 team-history metrics produced by build_team_history.
HIST_COLS = [
    "Hist_Pts_k",
    "Hist_W_k",
    "Hist_D_k",
    "Hist_L_k",
    "Hist_GF_k",
    "Hist_GA_k",
    "Hist_GD_k",
]


def team_match_long(frame):
    """One row per (team, match), with goals and points from that view.

    Parameters
    ----------
    frame : pandas.DataFrame
        Match-level frame with Season, Date_dt, HomeTeam, AwayTeam,
        FTHG, FTAG and FTR.

    Returns
    -------
    pandas.DataFrame
        Long frame with one row per team per match, sorted by Team,
        Season and Date_dt, carrying GF, GA, Pts, W, D and L from that
        team's own view.
    """

    def stack(is_home):
        team_col, other_col = (
            ("HomeTeam", "AwayTeam") if is_home else ("AwayTeam", "HomeTeam")
        )
        goals_for, goals_against = (
            ("FTHG", "FTAG") if is_home else ("FTAG", "FTHG")
        )
        stacked = frame[
            ["Season", "Date_dt", team_col, other_col, goals_for,
             goals_against, "FTR"]
        ].copy()
        stacked.columns = [
            "Season", "Date_dt", "Team", "Opponent", "GF", "GA", "FTR"
        ]
        stacked["is_home"] = is_home
        won = ((stacked["FTR"] == "H") & is_home) | (
            (stacked["FTR"] == "A") & (not is_home)
        )
        stacked["Pts"] = np.where(
            stacked["FTR"] == "D", 1, np.where(won, 3, 0)
        )
        stacked["W"] = (stacked["Pts"] == 3).astype(int)
        stacked["D"] = (stacked["Pts"] == 1).astype(int)
        stacked["L"] = (stacked["Pts"] == 0).astype(int)
        return stacked

    return (
        pd.concat([stack(True), stack(False)])
        .sort_values(["Team", "Season", "Date_dt"])
        .reset_index(drop=True)
    )


def past_roll(long_frame, column, how="sum", k=K):
    """Shift by one, then roll: strictly historical, per season.

    Parameters
    ----------
    long_frame : pandas.DataFrame
        Long team-match frame from team_match_long.
    column : str
        Column aggregated over the window.
    how : str
        Rolling aggregation, e.g. "sum".
    k : int
        Window size, in matches.

    Returns
    -------
    pandas.Series
        Aggregation of the k matches strictly before each row,
        aligned with long_frame's index.
    """
    shifted = long_frame.groupby(["Team", "Season"])[column].shift(1)
    return (
        shifted.groupby([long_frame["Team"], long_frame["Season"]])
        .rolling(k, min_periods=1)
        .agg(how)
        .reset_index(level=[0, 1], drop=True)
    )


def build_team_history(frame, k=K, cols=HIST_COLS):
    """Attach Home_/Away_/Diff_ history columns to a match-level frame.

    Every value comes from fixtures strictly earlier than the row being
    built, because past_roll shifts before it rolls. Idempotent: it
    drops its own columns first, so it can be called repeatedly.

    Parameters
    ----------
    frame : pandas.DataFrame
        Match-level frame with Season, Date, HomeTeam, AwayTeam,
        FTHG, FTAG and FTR.
    k : int
        Window size, in matches.
    cols : list of str
        History metrics to attach, from HIST_COLS.

    Returns
    -------
    pandas.DataFrame
        Same rows and index as frame, plus the Home_/Away_/Diff_ columns.
    """
    rows_before = len(frame)
    frame = frame.copy()
    original_index = frame.index

    if "Date_dt" not in frame.columns:
        frame["Date_dt"] = pd.to_datetime(
            frame["Date"], format="%d/%m/%Y"
        )

    long_frame = team_match_long(frame)

    for column in ["Pts", "W", "D", "L", "GF", "GA"]:
        long_frame["Hist_" + column + "_k"] = past_roll(
            long_frame, column, "sum", k
        )

    long_frame["Hist_GD_k"] = (
        long_frame["Hist_GF_k"] - long_frame["Hist_GA_k"]
    )

    family = [p + c for p in ("Home_", "Away_", "Diff_") for c in cols]
    frame = frame.drop(
        columns=[c for c in family if c in frame.columns], errors="ignore"
    )

    home = (
        long_frame[long_frame["is_home"]]
        [["Season", "Date_dt", "Team"] + cols]
        .rename(
            columns={
                "Team": "HomeTeam",
                **{c: "Home_" + c for c in cols},
            }
        )
    )
    away = (
        long_frame[~long_frame["is_home"]]
        [["Season", "Date_dt", "Team"] + cols]
        .rename(
            columns={
                "Team": "AwayTeam",
                **{c: "Away_" + c for c in cols},
            }
        )
    )

    frame = frame.merge(
        home, on=["Season", "Date_dt", "HomeTeam"],
        how="left", validate="m:1",
    )
    frame = frame.merge(
        away, on=["Season", "Date_dt", "AwayTeam"],
        how="left", validate="m:1",
    )
    difference = pd.DataFrame(
        {
            "Diff_" + c: frame["Home_" + c] - frame["Away_" + c]
            for c in cols
        }
    )
    frame = pd.concat([frame, difference], axis=1)

    if len(frame) != rows_before:
        raise ValueError("the merge changed the row count")
    openers = long_frame.groupby(["Team", "Season"]).head(1)
    if not openers["Hist_Pts_k"].isna().all():
        raise ValueError("a season opener should have no history")
    frame.index = original_index
    return frame


print(f"Historical block ready: {len(HIST_COLS)} metrics, K={K}.")


Historical block ready: 7 metrics, K=5.


### 2.9 Las 23 features

`build_features` es pura: no aprende medianas ni categorías y no necesita la etiqueta de las filas que transforma. Combina historial, probabilidades de mercado, calendario e identidad de los clubes. Se ejecuta una vez sobre el marco completo y luego se corta por índice.

**De dónde salen las 23.** 12 históricas + 5 de mercado + 3 de calendario + 1 de día + 2 de identidad de clubes.

| Grupo | N | Columnas | Qué aporta |
| --- | --- | --- | --- |
| Historial | 12 | `HIST_KEEP` | Forma, ataque y defensa de cada equipo, y su diferencia |
| Mercado | 5 | `p_home`, `p_draw`, `p_away`, `Mkt_Gap`, `AHh` | La probabilidad que el mercado asigna a cada resultado |
| Calendario | 3 | `PriorHomeMatches`, `PriorAwayMatches`, `OddsQuotes` | Momento de la temporada y cobertura de cuotas |
| Día | 1 | `DayOfWeek` | Patrón de descanso y programación |
| Clubes | 2 | `HomeTeam`, `AwayTeam` | Identidad del local y del visitante |

**Mercado, una por una.** `p_home`, `p_draw` y `p_away` son las probabilidades implícitas en `AvgH`, `AvgD` y `AvgA`, normalizadas para que sumen 1 (de-vig). `Mkt_Gap = AvgH - AvgA` mide la brecha de precios: al ser cuotas, un valor negativo indica que el local está más barato y por tanto es favorito. `AHh` es la línea de hándicap asiático: valores negativos también favorecen al local y el EDA la señaló como la variable pre-partido más informativa.

**Calendario, una por una.** `PriorHomeMatches` y `PriorAwayMatches` cuentan los partidos que cada equipo ya jugó en esa temporada (0 a 37), sin distinguir localía: sirven como etapa de la temporada y proxy de congestión. `OddsQuotes` cuenta cuántas de las 42 cuotas de intercambio están disponibles (0 a 42): mide cobertura del mercado, no el precio.

**Día y clubes.** `DayOfWeek` entra como one-hot. `HomeTeam` y `AwayTeam` también: con 26 clubes vistos en train son 52 columnas, y `handle_unknown="ignore"` permite que un club nuevo en test pierda solo su indicador. Sunderland aparece únicamente en 2025-26 y 14 columnas de clubes quedan constantes en test porque siete equipos descendieron.

**Antes y después del one-hot.** Las 23 son la entrada del preprocesador. Tras imputar, escalar y codificar clubes y día, la salida son 79 columnas.

**Lo que no entra.** `FTHG` y `FTAG` no llegan como columnas: solo alimentan el historial de partidos anteriores. Las estadísticas post-partido, las casas individuales, `Max*` y las cuotas de cierre quedan fuera por leakage o redundancia, según el inventario del notebook 02.

In [10]:
def matches_played(frame, side):
    """Count every prior league match of the side's team in its season.

    Home and away fixtures both count: the value is the team's
    matchday index inside the season, not its home-match index. A
    season opener starts at 0 and the last matchday reaches 37.

    Parameters
    ----------
    frame : pandas.DataFrame
        Match-level frame with Season, Date_dt, Time and the side
        column.
    side : str
        "HomeTeam" or "AwayTeam".

    Returns
    -------
    pandas.Series
        Matches already played by that team in that season before
        each row, aligned with frame's index.
    """
    if side not in ("HomeTeam", "AwayTeam"):
        raise ValueError(f"side must be HomeTeam or AwayTeam: {side}")
    order = frame.sort_values(["Season", "Date_dt", "Time"]).index
    long_frame = team_match_long(frame.loc[order])
    long_frame["Played"] = long_frame.groupby(["Team", "Season"]).cumcount()
    is_side = (
        long_frame["is_home"] if side == "HomeTeam" else ~long_frame["is_home"]
    )
    counts = long_frame.loc[is_side, ["Season", "Date_dt", "Team", "Played"]]
    merged = (
        frame.loc[order, ["Season", "Date_dt", side]]
        .merge(
            counts,
            left_on=["Season", "Date_dt", side],
            right_on=["Season", "Date_dt", "Team"],
            how="left",
            validate="m:1",
        )
    )
    if merged["Played"].isna().any():
        raise ValueError("a match found no count row in the long table")
    return pd.Series(merged["Played"].to_numpy(), index=order).reindex(
        frame.index
    )


def build_features(frame, k=K):
    """Build the 23 pipeline features from the raw allowed columns.

    Pure and stateless: no median and no category is learned, and the
    label of the rows being transformed is never needed. FTR is read
    only as the result of earlier matches, because the history block
    shifts before it rolls inside (Team, Season). The same call works
    on the full frame or on any subset of it.

    Parameters
    ----------
    frame : pandas.DataFrame
        Raw match frame including FTR.
    k : int
        Size of the recent-form window.

    Returns
    -------
    pandas.DataFrame
        Same rows and index as frame, with the PIPELINE_FEATURES
        columns.
    """
    needed = ALLOWED_RAW_COLUMNS + [TARGET_COLUMN]
    missing = [column for column in needed if column not in frame.columns]
    if missing:
        raise ValueError(f"missing required columns: {missing}")

    out = build_team_history(frame, k=k)
    probabilities = market_probabilities(out)
    out["p_home"] = probabilities[:, 0]
    out["p_draw"] = probabilities[:, 1]
    out["p_away"] = probabilities[:, 2]
    out["Mkt_Gap"] = out["AvgH"] - out["AvgA"]
    out["DayOfWeek"] = out["Date_dt"].dt.dayofweek
    out["PriorHomeMatches"] = matches_played(out, "HomeTeam")
    out["PriorAwayMatches"] = matches_played(out, "AwayTeam")
    out["OddsQuotes"] = out[ODDS_QUOTE_COLS].notna().sum(axis=1)

    absent = [column for column in PIPELINE_FEATURES if column not in out.columns]
    if absent:
        raise ValueError(f"features not built: {absent}")
    return out[PIPELINE_FEATURES].copy()


### 2.10 Features y particiones

Se generan las features y se alinean con las particiones. Las comprobaciones cubren: mismas filas, 23 columnas, `FTR` fuera de `X`, 1520/380 filas y que los contadores de partidos previos van de 0 a 37 por temporada. Lo último detectaría un historial que se leyó a sí mismo.

In [11]:
built = build_features(df)

assert built.index.equals(df.index), "feature rows must align with df"
assert len(built) == len(df)
assert "FTR" not in built.columns
assert set(built.columns) == set(PIPELINE_FEATURES)

X_train = built.loc[df_train.index]
X_test = built.loc[df_test.index]
y_train = df_train[TARGET_COLUMN]
y_test = df_test[TARGET_COLUMN]

assert len(X_train) == TRAIN_ROWS
assert len(X_test) == TEST_ROWS

arsenal = df["Season"] == "2021-2022"
arsenal_home = built.loc[arsenal & (df["HomeTeam"] == "Arsenal"), "PriorHomeMatches"]
arsenal_away = built.loc[arsenal & (df["AwayTeam"] == "Arsenal"), "PriorAwayMatches"]
# Arsenal opened 2021-22 away: the away opener has 0 prior matches and
# the first home match, on matchday 2, has 1.
assert arsenal_away.min() == 0 and arsenal_home.min() == 1
assert arsenal_home.max() == 37  # closed the season at home, matchday 38

# Both venues count: for every (Season, Team) the 38 prior-match counts
# are exactly 0..37 - no gaps, no repeats, venue-blind.
both = pd.concat(
    [
        built.assign(
            Season=df["Season"],
            Team=built["HomeTeam"],
            played=built["PriorHomeMatches"],
        )[["Season", "Team", "played"]],
        built.assign(
            Season=df["Season"],
            Team=built["AwayTeam"],
            played=built["PriorAwayMatches"],
        )[["Season", "Team", "played"]],
    ]
)
counts = both.groupby(["Season", "Team"])["played"].agg(
    ["size", "nunique", "min", "max"]
)
assert (counts["size"] == 38).all()
assert (counts["nunique"] == 38).all()
assert (counts["min"] == 0).all() and (counts["max"] == 37).all()

print(f"Built {built.shape[0]} rows x {built.shape[1]} features")
print(f"Train {X_train.shape}, test {X_test.shape}")
X_train.head()


Built 1900 rows x 23 features
Train (1520, 23), test (380, 23)


,Home_Hist_Pts_k,Away_Hist_Pts_k,Diff_Hist_Pts_k,Home_Hist_GF_k,Away_Hist_GF_k,Diff_Hist_GF_k,Home_Hist_GA_k,Away_Hist_GA_k,Home_Hist_W_k,Away_Hist_W_k,...,p_draw,p_away,Mkt_Gap,AHh,PriorHomeMatches,PriorAwayMatches,OddsQuotes,DayOfWeek,HomeTeam,AwayTeam
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.281591,0.478147,2.00,0.50,0,0,0,4,Brentford,Arsenal
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.214891,0.164005,-4.32,-1.00,0,0,0,5,Man United,Leeds
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.311564,0.386639,0.70,0.25,0,0,0,5,Burnley,Brighton
3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.162314,0.075070,-11.54,-1.50,0,0,0,5,Chelsea,Crystal Palace
4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.272804,0.238366,-2.07,-0.50,0,0,0,5,Everton,Southampton


Hay que ver `1900 × 23` features y las formas de train/test. Si los contadores de partidos previos no empiezan en 0 o no llegan a 37, el bloque histórico está mal alineado.

### 2.11 Transformaciones

Se definen los dos pipelines que aprenderán estadísticas: mediana + escalado para números, y one-hot para clubes y día de la semana. `handle_unknown="ignore"` permite que un club nuevo en test no rompa la predicción; solo pierde su indicador.

In [12]:
numeric_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]
)

categorical_pipeline = Pipeline(
    steps=[
        (
            "encoder",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=False,
            ),
        ),
    ]
)

print("Numeric and categorical pipelines built.")

Numeric and categorical pipelines built.


### 2.12 ColumnTransformer

Cada bloque recibe su grupo de columnas y su transformación. `remainder="drop"` garantiza que ninguna columna fuera del contrato llegue al modelo. El objeto se crea sin ajustar: se ajustará dentro de cada fold.

In [13]:
def build_preprocessor():
    """Build the ColumnTransformer described in section 7.

    Returns
    -------
    ColumnTransformer
        Unfitted transformer: median imputation and standardisation
        for the numeric groups, one-hot with handle_unknown="ignore"
        for clubs and day of week.
    """
    preprocessor = ColumnTransformer(
        transformers=[
            ("historical", numeric_pipeline, HIST_KEEP),
            ("market", numeric_pipeline, MARKET_KEEP),
            ("calendar", numeric_pipeline, CAL_KEEP),
            ("clubs", categorical_pipeline, CLUB_COLS),
            ("day_of_week", categorical_pipeline, DOW_KEEP),
        ],
        remainder="drop",
        verbose_feature_names_out=False,
    )
    return preprocessor.set_output(transform="pandas")


preprocessor = build_preprocessor()

print("ColumnTransformer built.")

ColumnTransformer built.


### 2.13 Pipeline sin ajustar

`build_model_pipeline` une preprocesamiento y estimador en un solo objeto. `clone` evita compartir estado entre experimentos y la comprobación final confirma que el estimador llega sin entrenar: el ajuste pertenece a la etapa de validación.

In [14]:
def build_model_pipeline(estimator):
    """Create an unfitted features-to-model pipeline.

    Parameters
    ----------
    estimator : estimator object
        Any scikit-learn estimator; cloned as the final "model" step.

    Returns
    -------
    Pipeline
        preprocessor -> model, all unfitted. Feed it the columns
        returned by build_features.
    """
    return Pipeline(
        steps=[
            ("preprocessor", build_preprocessor()),
            ("model", clone(estimator)),
        ]
    )


model_pipeline = build_model_pipeline(DummyClassifier())

print([name for name, _ in model_pipeline.steps])
assert not hasattr(model_pipeline.steps[-1][1], "classes_")
print("Estimator arrives unfitted; the modeling notebook owns the fit.")


['preprocessor', 'model']
Estimator arrives unfitted; the modeling notebook owns the fit.


### 2.14 Contrato ejecutable

Estos asserts son el contrato con el notebook 02: semilla, corte, tamaños, temporadas, columnas, etiquetas y objetos sin ajustar. Si el 02 cambia, esta celda falla y evita comparar modelos sobre particiones distintas.

In [15]:
# Contrato ejecutable con el notebook 02.
assert RANDOM_STATE == 777
assert CUTOFF == pd.Timestamp("2025-08-01")
assert (len(X_train), len(X_test)) == (TRAIN_ROWS, TEST_ROWS) == (1520, 380)
assert X_train.index.equals(df_train.index)
assert X_test.index.equals(df_test.index)
assert set(X_train.columns) == set(PIPELINE_FEATURES)
assert len(PIPELINE_FEATURES) == 23
assert not set(PIPELINE_FEATURES) & set(POST_MATCH_COLUMNS)
assert "FTR" not in X_train.columns
assert set(y_train.unique()) <= set(CLASSES)
assert sorted(df_train["Season"].unique()) == [
    "2021-2022", "2022-2023", "2023-2024", "2024-2025"
]
assert sorted(df_test["Season"].unique()) == ["2025-2026"]
assert not hasattr(preprocessor, "transformers_")  # nada ajustado todavía
assert len(
    build_model_pipeline(DummyClassifier()).steps
) == 2

print("Contrato 02 verificado: split, features y pipelines sin ajustar.")

Contrato 02 verificado: split, features y pipelines sin ajustar.


Si todo pasa, las secciones siguientes pueden confiar en `X_train`, `X_test`, `FOLDS` y en los pipelines sin volver a comprobar el origen de los datos.

## 3. Protocolo de validación cruzada

**Esquema: ventana creciente por temporada.** Cada simulacro entrena con las temporadas ya completas y valida con la siguiente. Es el único esquema que reproduce lo que se hará en test: predecir una temporada futura con lo que ya pasó.

| Fold | Entrena con | Valida con | Filas train | Filas validación |
| --- | --- | --- | --- | --- |
| 1 | 2021-22 | 2022-23 | 380 | 380 |
| 2 | 2021-22 a 2022-23 | 2023-24 | 760 | 380 |
| 3 | 2021-22 a 2023-24 | 2024-25 | 1140 | 380 |

**Por qué 3 folds.** Hay cuatro temporadas de train; la primera solo puede ser entrenamiento, así que quedan tres validaciones, cada una de una temporada completa (380 filas). Así la nota de cada fold pesa igual que el test real. Se descartan:

- `KFold` y `StratifiedKFold`: mezclan pasado y futuro y dan una nota optimista;
- `GroupKFold` por equipo: los clubes se repiten cada temporada, que es justo lo que se quiere generalizar;
- un único holdout: una sola nota puede ser suerte;
- 6 folds de media temporada: validaciones de 190 filas y más ruido, mientras el test es a nivel temporada.

**Mismos splits.** La lista se construye una sola vez y se pasa como el mismo objeto a todos los modelos. Los folds temporales no usan azar, así que no consumen semilla; la semilla 777 queda para los estimadores que sí son estocásticos.

**Qué se reporta.** Media y desviación estándar del log-loss entre los 3 folds. La desviación sale de tres números, así que diferencias pequeñas no deben sobreinterpretarse. El fold 1 entrena con solo 380 filas y será el más débil por construcción.

**Límite conocido:** la temporada 2021-22 nunca se valida. Es el precio de tener cuatro temporadas: se usa para que el primer fold no entrene con cero datos.

### 3.1 Construcción de los folds

Se construyen tres pares de posiciones con ventana creciente: cada fold entrena con las temporadas anteriores y valida con una temporada completa. Los asserts verifican tamaños, ausencia de solapamiento y que el entrenamiento termina antes de la validación. La tabla muestra el corte exacto de cada simulacro.

In [16]:
VALIDATION_SEASONS = sorted(df_train["Season"].unique())[1:]


def build_temporal_folds(frame, validation_seasons):
    """Pares (posiciones_train, posiciones_validacion) por temporada.

    Ventana creciente: cada fold entrena con todo lo anterior y valida
    con una temporada completa. Posiciones sobre `frame`, que debe
    estar alineado con X_train.
    """
    positions = np.arange(len(frame))
    seasons = frame["Season"].to_numpy()
    folds = []
    for season in validation_seasons:
        folds.append((
            positions[seasons < season],
            positions[seasons == season],
        ))
    return folds


FOLDS = build_temporal_folds(df_train, VALIDATION_SEASONS)

assert len(FOLDS) == 3
assert [len(train) for train, _ in FOLDS] == [380, 760, 1140]
assert [len(val) for _, val in FOLDS] == [380, 380, 380]

for (train_positions, val_positions), season in zip(FOLDS, VALIDATION_SEASONS):
    assert set(df_train.iloc[val_positions]["Season"]) == {season}
    assert set(train_positions).isdisjoint(val_positions)
    assert (
        df_train.iloc[train_positions]["Date"].max()
        < df_train.iloc[val_positions]["Date"].min()
    )

assert sorted(df_train.iloc[np.concatenate([val for _, val in FOLDS])]["Season"].unique()) == list(VALIDATION_SEASONS)
assert "2025-2026" not in set(df_train["Season"])

fold_table = pd.DataFrame([
    {
        "fold": number,
        "entrena_con": (
            f"{df_train.iloc[train]['Season'].min()}"
            f" a {df_train.iloc[train]['Season'].max()}"
        ),
        "valida_con": season,
        "filas_train": len(train),
        "filas_validacion": len(val),
        "último_partido_train": df_train.iloc[train]["Date"].max().date(),
        "primer_partido_validacion": df_train.iloc[val]["Date"].min().date(),
    }
    for number, ((train, val), season) in enumerate(
        zip(FOLDS, VALIDATION_SEASONS), start=1
    )
])

fold_table

,fold,entrena_con,valida_con,filas_train,filas_validacion,último_partido_train,primer_partido_validacion
0,1,2021-2022 a 2021-2022,2022-2023,380,380,2022-05-22,2022-08-05
1,2,2021-2022 a 2022-2023,2023-2024,760,380,2023-05-28,2023-08-11
2,3,2021-2022 a 2023-2024,2024-2025,1140,380,2024-05-19,2024-08-16


### 3.2 Lectura de la tabla

Los tres folds validan 380 partidos cada uno, con 380, 760 y 1140 filas de entrenamiento. El primero es el más débil por construcción; por eso se reportan media y desviación, no solo la mejor nota. La temporada 2021-22 solo entrena: no hay fold que la valide.

## 4. Baselines

Referencia ingenua y referencia de mercado como pisos a superar. No compiten con los modelos: enmarcan el resultado.

In [17]:
# TODO: baseline ingenuo (DummyClassifier)
# TODO: referencia de mercado (de-vig)
# TODO: interpretación de ambos pisos

## 5. Modelos candidatos

Al menos tres modelos distintos y apropiados para el problema. Para cada uno: por qué podría funcionar, qué preparación necesita, ventajas y limitaciones, y configuración inicial.

In [18]:
# TODO: definir al menos tres modelos candidatos
# TODO: para cada uno: por qué podría funcionar
# TODO: preparación que necesita
# TODO: ventajas y limitaciones
# TODO: configuración inicial

## 6. Comparación inicial

Tabla comparable (media, desviación, tiempo) e interpretación: qué diferencia es real y cuál cabe dentro de la variación entre folds. Decidir qué modelos pasan a tuning.

In [19]:
# TODO: evaluar todos los candidatos con los mismos folds
# TODO: tabla modelo / configuración / media / desviación / tiempo
# TODO: interpretación y decisión de qué pasa a tuning

## 7. Optimización con Optuna

Función objetivo, espacios de búsqueda acotados, validación cruzada dentro de cada trial, presupuesto explícito, sampler y semilla. Comparar el mejor trial contra la configuración inicial.

In [20]:
# TODO: función objetivo y dirección de optimización
# TODO: espacio de búsqueda por modelo
# TODO: validación cruzada dentro de cada trial
# TODO: presupuesto (trials y/o tiempo)
# TODO: sampler y semilla

### 7.1 Ejecución de los estudios

Se ejecutan los estudios con el presupuesto y el sampler definidos arriba. La salida debe comparar el mejor trial de cada modelo contra su configuración inicial, no solo mostrar el número ganador.

In [21]:
# TODO: ejecutar los estudios
# TODO: mejores parámetros por modelo

## 8. Comparación inicial vs. ajustado

Misma tabla y mismas métricas para ambos escenarios. Responder si el tuning produjo una mejora relevante frente a la varianza.

In [22]:
# TODO: recalcular los modelos ajustados con el mismo protocolo
# TODO: tabla comparativa inicial vs. ajustado
# TODO: interpretación de la mejora

## 9. Selección final (solo CV)

Criterio explícito: rendimiento, variación entre folds, costo y complejidad. Test sigue sin participar.

In [23]:
# TODO: criterio de selección (rendimiento, varianza, costo, complejidad)
# TODO: pipeline final elegido sin mirar test

## 10. Ajuste final y evaluación única en test

Ajustar el pipeline elegido con todo train y predecir test una sola vez. Matriz de confusión y métricas por clase.

In [24]:
# TODO: ajustar el pipeline final con todo train
# TODO: predicción única sobre test
# TODO: matriz de confusión
# TODO: métricas por clase

## 11. Análisis de errores y limitaciones

Dónde falla el modelo y qué limitaciones conserva. El resultado de test describe generalización; no se usa para volver a decidir.

In [25]:
# TODO: análisis de errores por clase y por temporada
# TODO: calibración / log-loss si aplica
# TODO: limitaciones heredadas del notebook 02

## 12. Conclusiones

Pipeline elegido y por qué, desempeño medio y variación, si el tuning aportó una mejora relevante y qué limitaciones quedan.

## Checklist antes de entregar

- [ ] Métrica principal definida.
- [ ] Esquema de validación cruzada justificado.
- [ ] Baseline ingenuo incluido.
- [ ] Al menos tres modelos comparados.
- [ ] Mismos folds para todos.
- [ ] Media y desviación reportadas.
- [ ] Sin leakage dentro de la validación cruzada.
- [ ] Uso o no de Optuna justificado.
- [ ] Comparación antes y después del tuning.
- [ ] Selección sin mirar test.
- [ ] Test evaluado una sola vez.
- [ ] Errores y limitaciones interpretados.
- [ ] Kernel reiniciado y Run All completado.